# CS 413 Affective Computing — Week 7 Lab
## Facial Emotion Recognition on FER2013 · Track B (frozen CNN features + classical classifiers)

| | |
|---|---|
| **Dataset** | FER2013: 48×48 grayscale faces, 7 emotions (`data/train/<class>/`, `data/test/<class>/`) |
| **Face detection** | MediaPipe BlazeFace (Haar cascade fallback) on images upscaled to 192×192 |
| **Features** | MobileNetV2, ImageNet weights, frozen, global-average pooled → 1280-d embedding |
| **Classifiers** | `StandardScaler` → Logistic Regression / Linear SVM, each in a scikit-learn `Pipeline` |
| **Reproducibility** | Seed 42 for `random`, NumPy, TensorFlow and every scikit-learn `random_state` |

**How to run:** install `requirements.txt`, place FER2013 under `data/`, then *Kernel → Restart & Run All*.
Embeddings are cached in `features/`, so re-runs skip the slow extraction step.
Figures and tables are written to `outputs/`, the final model to `models/`.

## Team roles

| Member | Role | Responsibilities |
|---|---|---|
| _Name 1_ | Data & preprocessing lead | Dataset checks, face detection (Milestone 1) |
| _Name 2_ | Feature extraction lead | MobileNetV2 embeddings, feature cache (Milestone 2) |
| _Name 3_ | Modelling & evaluation lead | Training, metrics, error analysis (Milestone 3) |
| _Name 4_ | Deployment & report lead | `predict_emotion`, demo, report write-up |

## 0. Setup & sanity check

Imports, configuration and seeding. `SAMPLE_PER_CLASS` sets the run size: `None` uses the full dataset,
a small number (e.g. `50`) samples that many images per class and split for a quick end-to-end run.

In [ ]:
import atexit
import json
import os
import platform
import random
import sys
import time
import urllib.request
from pathlib import Path

# TensorFlow 2.21 and MediaPipe print C++ start-up logs straight to the process stderr, which Jupyter shows as
# red blocks (TF_CPP_MIN_LOG_LEVEL no longer silences them). Inside Jupyter, send that native stream to
# logs/native_stderr.log. Python warnings and errors don't use it and still appear in the notebook.
if "ipykernel" in sys.modules:
    Path("logs").mkdir(exist_ok=True)
    _native_log = open("logs/native_stderr.log", "a")
    os.dup2(_native_log.fileno(), 2)

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import tensorflow as tf
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_recall_fscore_support)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from tensorflow.keras.applications import mobilenet_v2
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
SEED = 42
SAMPLE_PER_CLASS = None  # None = full dataset; e.g. 50 → quick run (images per class, per split)

DATA_DIR = Path("data")
FEATURES_DIR, OUTPUT_DIR, MODELS_DIR = Path("features"), Path("outputs"), Path("models")
CLASS_NAMES = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]  # FER2013 label ids 0–6
SPLITS = ["train", "test"]
IMAGE_EXTS = {".png", ".jpg", ".jpeg"}

DETECT_SIZE = 192     # upscale 48×48 → 192×192 before face detection
DETECT_MARGIN = 0.25  # black border (fraction of image size) so tightly cropped faces don't touch the frame edge
CROP_PAD = 0.10       # padding around the detected box when cropping
CNN_SIZE = 224        # MobileNetV2 input resolution
BATCH_SIZE = 64

for folder in (FEATURES_DIR, OUTPUT_DIR, MODELS_DIR):
    folder.mkdir(exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Plot style: colour-blind-safe blue/orange pair, single-hue blue ramp for heatmaps ──
C_BLUE, C_ORANGE, C_INK, C_MUTED, C_GOOD, C_BAD = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#0ca30c", "#d03b3b"
HEAT_CMAP = LinearSegmentedColormap.from_list(
    "blue_ramp", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
sns.set_theme(style="whitegrid", rc={"grid.color": "#e1e0d9", "axes.edgecolor": "#c3c2b7",
                                     "text.color": C_INK, "axes.labelcolor": C_MUTED,
                                     "xtick.color": C_MUTED, "ytick.color": C_MUTED})


def save_fig(fig, name):
    """Save a figure to outputs/ (PNG, 150 dpi) for the report."""
    path = OUTPUT_DIR / name
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Saved → {path}")


def clean_ax(ax):
    """Hide ticks, gridlines and spines on an image panel (axis labels stay usable)."""
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)


def load_gray(path):
    """Read an image file as a single-channel uint8 array."""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise IOError(f"Could not read image: {path}")
    return img


print(f"Python {platform.python_version()} | TensorFlow {tf.__version__} | OpenCV {cv2.__version__} | "
      f"scikit-learn {sklearn.__version__} | NumPy {np.__version__} | pandas {pd.__version__}")
print(f"SEED = {SEED} | SAMPLE_PER_CLASS = {SAMPLE_PER_CLASS}")

### 0.1 Class folders and image counts

In [ ]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"'{DATA_DIR}/' not found: expected data/train/<class>/ and data/test/<class>/")

missing = [f"{split}/{c}" for split in SPLITS for c in CLASS_NAMES if not (DATA_DIR / split / c).is_dir()]
if missing:
    raise FileNotFoundError(f"Missing class folders: {missing}")
print("✓ All 7 class folders present in both data/train/ and data/test/")

for split in SPLITS:
    extra = sorted(p.name for p in (DATA_DIR / split).iterdir() if p.is_dir() and p.name not in CLASS_NAMES)
    if extra:
        print(f"  note: ignoring unexpected folders in {split}/: {extra}")


def list_images(folder):
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTS)


image_paths = {split: {c: list_images(DATA_DIR / split / c) for c in CLASS_NAMES} for split in SPLITS}
empty = [f"{split}/{c}" for split in SPLITS for c in CLASS_NAMES if not image_paths[split][c]]
if empty:
    raise ValueError(f"Class folders without images: {empty}")

counts = pd.DataFrame({split: [len(image_paths[split][c]) for c in CLASS_NAMES] for split in SPLITS},
                      index=pd.Index(CLASS_NAMES, name="class"))
counts["total"] = counts["train"] + counts["test"]
counts.loc["TOTAL"] = counts.sum()
counts["train %"] = (100 * counts["train"] / counts.loc["TOTAL", "train"]).round(1)
counts.to_csv(OUTPUT_DIR / "class_counts.csv")
display(counts)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
x, width = np.arange(len(CLASS_NAMES)), 0.38
for offset, split, color in [(-0.5, "train", C_BLUE), (0.5, "test", C_ORANGE)]:
    bars = ax.bar(x + offset * width, counts.loc[CLASS_NAMES, split], width=width,
                  color=color, label=split, edgecolor="white", linewidth=1.5)
    ax.bar_label(bars, fmt="{:,.0f}", padding=2, fontsize=8, color=C_MUTED)
ax.set_xticks(x, CLASS_NAMES)
ax.set_ylabel("Images")
ax.set_title("FER2013 images per class and split")
ax.grid(axis="x", visible=False)
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
sns.despine(left=True)
save_fig(fig, "class_distribution.png")
plt.show()

### 0.2 Sample images per class

In [ ]:
N_SAMPLES = 6
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(len(CLASS_NAMES), N_SAMPLES, figsize=(N_SAMPLES * 1.45, len(CLASS_NAMES) * 1.5),
                         layout="constrained")
for r, cls in enumerate(CLASS_NAMES):
    files = image_paths["train"][cls]
    picks = rng.choice(len(files), size=min(N_SAMPLES, len(files)), replace=False)
    for c, ax in enumerate(axes[r]):
        clean_ax(ax)
        if c < len(picks):
            ax.imshow(load_gray(files[picks[c]]), cmap="gray", vmin=0, vmax=255)
    axes[r, 0].set_ylabel(cls, rotation=0, ha="right", va="center", fontsize=11, color=C_INK)
fig.suptitle("Random training samples per class (48×48)")
save_fig(fig, "sample_grid.png")
plt.show()

## 1. Milestone 1 — Face detection & visual check

FER2013 faces are only 48×48 and already tightly cropped, while face detectors are trained on larger photos with
some background around the face, so they often fail at native resolution. For every image we therefore:

1. **Upscale** to 192×192 (bicubic).
2. Add a **black margin** (25% of the image size) so the face no longer touches the image border.
3. Run **MediaPipe Face Detection** (Tasks API, BlazeFace short-range model). If MediaPipe can't be imported or
   initialised, fall back to OpenCV's **Haar cascade**.
4. Crop the **largest** face with 10% padding. If **no face is found, use the full image** and count the fallback.

### 1.1 Detector setup

In [ ]:
MP_MODEL_URL = ("https://storage.googleapis.com/mediapipe-models/face_detector/"
                "blaze_face_short_range/float16/latest/blaze_face_short_range.tflite")
MP_MODEL_PATH = MODELS_DIR / "blaze_face_short_range.tflite"


def build_face_detector():
    """Return (name, detect) where detect(gray_uint8) -> list of (x, y, w, h) face boxes in pixels."""
    try:
        import mediapipe as mp
        from mediapipe.tasks.python import BaseOptions, vision

        if not MP_MODEL_PATH.exists():  # one-time download of the ~230 KB model file
            tmp_path = MP_MODEL_PATH.with_suffix(".part")
            urllib.request.urlretrieve(MP_MODEL_URL, tmp_path)
            tmp_path.rename(MP_MODEL_PATH)
        detector = vision.FaceDetector.create_from_options(vision.FaceDetectorOptions(
            base_options=BaseOptions(model_asset_path=str(MP_MODEL_PATH)),
            min_detection_confidence=0.5))
        atexit.register(detector.close)  # release native resources before the interpreter shuts down

        def detect(gray):
            rgb = np.ascontiguousarray(cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB))
            result = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
            return [(d.bounding_box.origin_x, d.bounding_box.origin_y,
                     d.bounding_box.width, d.bounding_box.height) for d in result.detections]

        return f"MediaPipe {mp.__version__} (BlazeFace short-range)", detect

    except Exception as err:  # import error, missing system library, model download failure, ...
        print(f"MediaPipe unavailable ({type(err).__name__}: {err}) → falling back to Haar cascade")
        cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

        def detect(gray):
            boxes = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(48, 48))
            return [tuple(int(v) for v in box) for box in boxes]

        return "OpenCV Haar cascade", detect


DETECTOR_NAME, detect_raw = build_face_detector()
print(f"Face detector: {DETECTOR_NAME}")

### 1.2 Reusable `detect_and_crop(img)`

In [ ]:
def upscale(gray):
    """Upscale so the shorter side is DETECT_SIZE (48×48 → 192×192); larger images are left unchanged."""
    scale = DETECT_SIZE / min(gray.shape[:2])
    if scale <= 1:
        return gray
    return cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)


def detect_faces(gray):
    """Upscale, add the margin, detect. Returns (upscaled image, face boxes in upscaled-image coordinates)."""
    up = upscale(gray)
    h, w = up.shape[:2]
    m = int(round(DETECT_MARGIN * max(h, w)))
    canvas = cv2.copyMakeBorder(up, m, m, m, m, cv2.BORDER_CONSTANT, value=0)
    boxes = []
    for x, y, bw, bh in detect_raw(canvas):
        x0, y0 = max(x - m, 0), max(y - m, 0)  # undo the margin and clip to the image
        x1, y1 = min(x - m + bw, w), min(y - m + bh, h)
        if min(x1 - x0, y1 - y0) >= 16:
            boxes.append((x0, y0, x1 - x0, y1 - y0))
    return up, boxes


def crop_largest(up, boxes):
    """Crop the largest box (by area) with CROP_PAD padding on each side."""
    x, y, w, h = max(boxes, key=lambda b: b[2] * b[3])
    px, py = int(CROP_PAD * w), int(CROP_PAD * h)
    return up[max(y - py, 0):y + h + py, max(x - px, 0):x + w + px]


def detect_and_crop(img):
    """Upscale → detect → crop the largest face (+ padding).

    Returns (face, detected). If no face is found, the full (upscaled) image is returned with detected=False.
    """
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    up, boxes = detect_faces(img)
    if not boxes:
        return up, False
    return crop_largest(up, boxes), True

### 1.3 Visual check: 3 random training images per class

Each pair shows the upscaled image with the detected box (green = largest face) and the crop that goes into
the CNN. Red titles mark images where no face was found and the full image is used instead.

In [ ]:
N_PER_CLASS = 3
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(len(CLASS_NAMES), 2 * N_PER_CLASS,
                         figsize=(2 * N_PER_CLASS * 1.55, len(CLASS_NAMES) * 1.75), layout="constrained")
for r, cls in enumerate(CLASS_NAMES):
    files = image_paths["train"][cls]
    picks = rng.choice(len(files), size=min(N_PER_CLASS, len(files)), replace=False)
    for k, idx in enumerate(picks):
        up, boxes = detect_faces(load_gray(files[idx]))
        vis = cv2.cvtColor(up, cv2.COLOR_GRAY2RGB)
        largest = max(boxes, key=lambda b: b[2] * b[3]) if boxes else None
        for (x, y, w, h) in boxes:
            color, thickness = ((12, 163, 12), 3) if (x, y, w, h) == largest else ((235, 104, 52), 1)
            cv2.rectangle(vis, (x, y), (x + w, y + h), color, thickness)
        face = crop_largest(up, boxes) if boxes else up

        ax_det, ax_crop = axes[r, 2 * k], axes[r, 2 * k + 1]
        ax_det.imshow(vis)
        ax_crop.imshow(face, cmap="gray", vmin=0, vmax=255)
        ax_det.set_title("face found" if boxes else "no face → full image",
                         fontsize=8, color=C_INK if boxes else C_BAD)
        ax_crop.set_title("crop → CNN", fontsize=8, color=C_MUTED)
    for ax in axes[r]:
        clean_ax(ax)
    axes[r, 0].set_ylabel(cls, rotation=0, ha="right", va="center", fontsize=11, color=C_INK)
fig.suptitle(f"Face detection on upscaled FER2013 images · {DETECTOR_NAME}")
save_fig(fig, "m1_face_detection_grid.png")
plt.show()

### 1.4 Detection rate on a sample of the training set

Up to 200 random training images per class (seeded).

In [ ]:
DETECTION_SAMPLE_PER_CLASS = 200
rng = np.random.default_rng(SEED)
rows = []
for cls in CLASS_NAMES:
    files = image_paths["train"][cls]
    picks = rng.choice(len(files), size=min(DETECTION_SAMPLE_PER_CLASS, len(files)), replace=False)
    found = sum(detect_and_crop(load_gray(files[i]))[1] for i in picks)
    rows.append({"class": cls, "sampled": len(picks), "face found": found, "fallback (full image)": len(picks) - found})
det_rate = pd.DataFrame(rows).set_index("class")
det_rate.loc["TOTAL"] = det_rate.sum()
det_rate["detection rate %"] = (100 * det_rate["face found"] / det_rate["sampled"]).round(1)
det_rate.to_csv(OUTPUT_DIR / "m1_detection_rate.csv")

print(f"Detector: {DETECTOR_NAME}")
print(f"Overall detection rate: {det_rate.loc['TOTAL', 'detection rate %']}% "
      f"({det_rate.loc['TOTAL', 'face found']:,} / {det_rate.loc['TOTAL', 'sampled']:,} sampled training images)")
display(det_rate)

## 2. Milestone 2 — Feature extraction

Per-image pipeline:

`load grayscale → detect_and_crop → resize 224×224 → stack to 3 channels → mobilenet_v2.preprocess_input → MobileNetV2 → 1280-d embedding`

MobileNetV2 uses ImageNet weights with `include_top=False, pooling='avg'` and is frozen: it is only used as a
feature extractor. Images go through the network in batches of 64.

### 2.1 Feature extractor

In [ ]:
feature_extractor = mobilenet_v2.MobileNetV2(weights="imagenet", include_top=False, pooling="avg",
                                             input_shape=(CNN_SIZE, CNN_SIZE, 3))
feature_extractor.trainable = False
EMBED_DIM = feature_extractor.output_shape[-1]
print(f"MobileNetV2 (ImageNet, frozen): input {feature_extractor.input_shape} → "
      f"output {feature_extractor.output_shape}, {feature_extractor.count_params():,} parameters")

In [ ]:
def preprocess_for_cnn(gray):
    """Grayscale image → (224×224×3 float32 array in [0, 255], face-detected flag)."""
    face, detected = detect_and_crop(gray)
    interp = cv2.INTER_AREA if min(face.shape[:2]) > CNN_SIZE else cv2.INTER_CUBIC
    face = cv2.resize(face, (CNN_SIZE, CNN_SIZE), interpolation=interp)
    return np.repeat(face[..., None], 3, axis=-1).astype(np.float32), detected


def embed(batch):
    """MobileNetV2 embeddings for a float32 batch (n, 224, 224, 3) with values in [0, 255]."""
    return feature_extractor.predict_on_batch(mobilenet_v2.preprocess_input(batch))


def extract_features(paths, desc):
    """Batched extraction. Returns X (n × 1280, float32) and per-image face-detected flags."""
    X = np.empty((len(paths), EMBED_DIM), dtype=np.float32)
    detected = np.empty(len(paths), dtype=bool)
    with tqdm(total=len(paths), desc=desc, unit="img", file=sys.stdout, mininterval=5) as pbar:
        for start in range(0, len(paths), BATCH_SIZE):
            chunk = paths[start:start + BATCH_SIZE]
            batch = np.empty((len(chunk), CNN_SIZE, CNN_SIZE, 3), dtype=np.float32)
            for i, path in enumerate(chunk):
                batch[i], detected[start + i] = preprocess_for_cnn(load_gray(path))
            X[start:start + len(chunk)] = embed(batch)
            pbar.update(len(chunk))
    return X, detected

### 2.2 Extract (or load cached) features

Features are cached in `features/` (`X_train.npy`, `y_train.npy`, `X_test.npy`, `y_test.npy`, plus the
per-image detection flags). `meta.json` stores the settings and file list the cache was built from. The cache is
reused only when both match the current run, so changing `SAMPLE_PER_CLASS` or a detection setting triggers a
fresh extraction automatically.

In [ ]:
def select_split(split):
    """Image paths and integer labels for one split (SAMPLE_PER_CLASS random images per class if set)."""
    rng = np.random.default_rng(SEED)
    paths, labels = [], []
    for label, cls in enumerate(CLASS_NAMES):
        files = image_paths[split][cls]
        if SAMPLE_PER_CLASS is not None and len(files) > SAMPLE_PER_CLASS:
            files = [files[i] for i in sorted(rng.choice(len(files), SAMPLE_PER_CLASS, replace=False))]
        paths += files
        labels += [label] * len(files)
    return paths, np.array(labels, dtype=np.int64)


train_paths, y_train = select_split("train")
test_paths, y_test = select_split("test")

cache_files = {name: FEATURES_DIR / f"{name}.npy"
               for name in ["X_train", "y_train", "X_test", "y_test", "detected_train", "detected_test"]}
meta_path = FEATURES_DIR / "meta.json"
cache_key = {
    "backbone": "MobileNetV2/imagenet/include_top=False/pooling=avg",
    "detector": DETECTOR_NAME, "detect_size": DETECT_SIZE, "detect_margin": DETECT_MARGIN,
    "crop_pad": CROP_PAD, "cnn_size": CNN_SIZE, "sample_per_class": SAMPLE_PER_CLASS, "seed": SEED,
    "train_paths": [p.as_posix() for p in train_paths], "test_paths": [p.as_posix() for p in test_paths],
}
cache_valid = (meta_path.exists() and all(f.exists() for f in cache_files.values())
               and json.loads(meta_path.read_text()) == cache_key)

if cache_valid:
    X_train, X_test = np.load(cache_files["X_train"]), np.load(cache_files["X_test"])
    detected_train, detected_test = np.load(cache_files["detected_train"]), np.load(cache_files["detected_test"])
    assert np.array_equal(np.load(cache_files["y_train"]), y_train)
    assert np.array_equal(np.load(cache_files["y_test"]), y_test)
    print(f"Loaded cached features from {FEATURES_DIR}/")
else:
    start_time = time.perf_counter()
    X_train, detected_train = extract_features(train_paths, "train")
    X_test, detected_test = extract_features(test_paths, "test")
    arrays = {"X_train": X_train, "y_train": y_train, "X_test": X_test, "y_test": y_test,
              "detected_train": detected_train, "detected_test": detected_test}
    for name, arr in arrays.items():
        np.save(cache_files[name], arr)
    meta_path.write_text(json.dumps(cache_key))  # written last: an interrupted run never leaves a "valid" cache
    print(f"Extracted {len(train_paths) + len(test_paths):,} images in {time.perf_counter() - start_time:.0f} s "
          f"→ cached to {FEATURES_DIR}/")

### 2.3 Train–test split verification

In [ ]:
assert X_train.shape == (len(y_train), EMBED_DIM) and X_test.shape == (len(y_test), EMBED_DIM)
assert np.isfinite(X_train).all() and np.isfinite(X_test).all()
assert not {p.as_posix() for p in train_paths} & {p.as_posix() for p in test_paths}, "train/test overlap"

print("Feature matrices")
print(f"  X_train: {X_train.shape}  {X_train.dtype}   y_train: {y_train.shape}  {y_train.dtype}")
print(f"  X_test : {X_test.shape}   {X_test.dtype}   y_test : {y_test.shape}   {y_test.dtype}")
print("No-face fallback (full image used)")
for split, flags in [("train", detected_train), ("test", detected_test)]:
    print(f"  {split:5}: {(~flags).sum():,} / {len(flags):,} images ({100 * (~flags).mean():.1f}%)")

label_map = pd.DataFrame({"class": CLASS_NAMES,
                          "train images": np.bincount(y_train, minlength=len(CLASS_NAMES)),
                          "test images": np.bincount(y_test, minlength=len(CLASS_NAMES))},
                         index=pd.Index(range(len(CLASS_NAMES)), name="label id"))
label_map.to_csv(OUTPUT_DIR / "label_encoding.csv")
print("\nLabel encoding")
display(label_map)

## 3. Model training

Both models use the same preprocessing: a `StandardScaler` (zero mean, unit variance per embedding dimension)
as the first step of the `Pipeline`, so its statistics come from the training data only.

* **Model 1:** `LogisticRegression(max_iter=2000, random_state=42)` (multinomial, L-BFGS).
* **Model 2:** `LinearSVC(random_state=42)`, a linear SVM (one-vs-rest, squared hinge loss). `SVC(kernel='linear')`
  scales roughly quadratically with ~29k training samples and would take far longer for the same kind of
  linear decision boundary.

**Class imbalance.** *disgust* has only about 1.5% of the training images. For each model we compare
`class_weight=None` and `'balanced'` on a **stratified 15% validation split of the training set** and keep the
setting with the higher macro-F1. The test set is never used for this choice. The chosen configuration is then
refit on the full training set.

### 3.1 Class-weight selection (validation split)

In [ ]:
MODEL_SPECS = {
    "Logistic Regression": lambda cw: LogisticRegression(max_iter=2000, class_weight=cw, random_state=SEED),
    "Linear SVM": lambda cw: LinearSVC(C=1.0, max_iter=5000, class_weight=cw, random_state=SEED),
}


def make_pipeline(name, class_weight=None):
    return Pipeline([("scaler", StandardScaler()), ("clf", MODEL_SPECS[name](class_weight))])


X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.15, stratify=y_train,
                                            random_state=SEED)
rows = []
for name in MODEL_SPECS:
    for cw in [None, "balanced"]:
        start_time = time.perf_counter()
        y_hat = make_pipeline(name, cw).fit(X_tr, y_tr).predict(X_val)
        rows.append({"model": name, "class_weight": str(cw),
                     "val accuracy": accuracy_score(y_val, y_hat),
                     "val macro-F1": f1_score(y_val, y_hat, average="macro", zero_division=0),
                     "fit time (s)": time.perf_counter() - start_time})
cw_results = pd.DataFrame(rows)

best_rows = cw_results.loc[cw_results.groupby("model", sort=False)["val macro-F1"].idxmax()]
CLASS_WEIGHT = {r["model"]: None if r["class_weight"] == "None" else r["class_weight"]
                for _, r in best_rows.iterrows()}
cw_results["selected"] = np.where(cw_results.index.isin(best_rows.index), "✓", "")
cw_results.round(4).to_csv(OUTPUT_DIR / "class_weight_selection.csv", index=False)

print(f"Train/validation split: {len(y_tr):,} / {len(y_val):,} samples (stratified)")
display(cw_results.round(4))
for name, cw in CLASS_WEIGHT.items():
    print(f"→ {name}: class_weight={cw!r}")

### 3.2 Final training on the full training set

In [ ]:
models, train_times = {}, {}
for name in MODEL_SPECS:
    start_time = time.perf_counter()
    models[name] = make_pipeline(name, CLASS_WEIGHT[name]).fit(X_train, y_train)
    train_times[name] = time.perf_counter() - start_time
    print(f"{name}: trained on {len(y_train):,} samples in {train_times[name]:.1f} s")

### 3.3 Model summaries

In [ ]:
SUMMARY_PARAMS = {"Logistic Regression": ["solver", "C", "max_iter", "class_weight", "random_state"],
                  "Linear SVM": ["loss", "C", "dual", "max_iter", "class_weight", "random_state"]}


def model_summary(name):
    pipe = models[name]
    clf = pipe.named_steps["clf"]
    params = clf.get_params()
    n_iter = int(np.max(clf.n_iter_))
    print(f"── {name} ──")
    print(f"Pipeline        : {' → '.join(type(step).__name__ for _, step in pipe.steps)}")
    print(f"Preprocessing   : StandardScaler on {X_train.shape[1]}-d MobileNetV2 embeddings (fit on train only)")
    print("Hyperparameters : " + ", ".join(f"{k}={params[k]!r}" for k in SUMMARY_PARAMS[name]))
    print(f"Iterations      : {n_iter} of max_iter={params['max_iter']}")
    print(f"Training time   : {train_times[name]:.1f} s on {len(y_train):,} samples")

In [ ]:
model_summary("Logistic Regression")

In [ ]:
model_summary("Linear SVM")

## 4. Milestone 3 — Evaluation

All metrics are computed on the held-out **test** split. FER2013 is imbalanced (*happy* is about 25% of the
images, *disgust* about 1.5%), so besides accuracy we report **macro** averages, which weight every emotion
equally. The best model is chosen by test macro-F1.

In [ ]:
def class_scores(model, X):
    """Per-class scores: predict_proba if available, otherwise a softmax over the SVM decision function."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)
    z = model.decision_function(X)
    z = np.exp(z - z.max(axis=1, keepdims=True))
    return z / z.sum(axis=1, keepdims=True)


def plot_confusion(cm, name, filename):
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6.4), layout="constrained")
    panels = [(axes[0], cm, "d", "counts", None), (axes[1], cm_norm, ".2f", "row-normalised (recall)", 1)]
    for ax, data, fmt, subtitle, vmax in panels:
        sns.heatmap(data, annot=True, fmt=fmt, cmap=HEAT_CMAP, vmin=0, vmax=vmax, square=True,
                    linewidths=1, linecolor="white", annot_kws={"fontsize": 9},
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
        ax.set(xlabel="Predicted", ylabel="True", title=f"{name}: {subtitle}")
        ax.tick_params(axis="x", rotation=45)
        ax.tick_params(axis="y", rotation=0)
    save_fig(fig, filename)
    plt.show()


results, predictions = {}, {}


def evaluate(name):
    y_pred = models[name].predict(X_test)
    predictions[name] = y_pred
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="macro", zero_division=0)
    results[name] = {"accuracy": acc, "macro_precision": prec, "macro_recall": rec, "macro_f1": f1}

    print(f"{name}\n  Accuracy : {acc:.4f}\n  Macro-F1 : {f1:.4f}\n")
    report = classification_report(y_test, y_pred, labels=range(len(CLASS_NAMES)), target_names=CLASS_NAMES,
                                   digits=3, zero_division=0)
    print(report)
    slug = name.lower().replace(" ", "_")
    (OUTPUT_DIR / f"classification_report_{slug}.txt").write_text(report)
    cm = confusion_matrix(y_test, y_pred, labels=range(len(CLASS_NAMES)))
    plot_confusion(cm, name, f"confusion_matrix_{slug}.png")

### 4.1 Logistic Regression

In [ ]:
evaluate("Logistic Regression")

### 4.2 Linear SVM

In [ ]:
evaluate("Linear SVM")

### 4.3 Model comparison

In [ ]:
comparison = pd.DataFrame(results).T
comparison.index.name = "model"
comparison["class_weight"] = [str(CLASS_WEIGHT[name]) for name in comparison.index]
comparison["train_time_s"] = [train_times[name] for name in comparison.index]
comparison.round(4).to_csv(OUTPUT_DIR / "model_comparison.csv")
BEST = comparison["macro_f1"].astype(float).idxmax()

display(comparison.round(4))
print(f"Best model (highest test macro-F1): {BEST}")

metrics = ["accuracy", "macro_precision", "macro_recall", "macro_f1"]
fig, ax = plt.subplots(figsize=(9, 4.2))
x, width = np.arange(len(metrics)), 0.36
for offset, (name, color) in zip([-0.5, 0.5], zip(comparison.index, [C_BLUE, C_ORANGE])):
    bars = ax.bar(x + offset * width, comparison.loc[name, metrics].astype(float), width=width,
                  color=color, label=name, edgecolor="white", linewidth=1.5)
    ax.bar_label(bars, fmt="{:.3f}", padding=2, fontsize=8, color=C_MUTED)
ax.set_xticks(x, ["Accuracy", "Macro precision", "Macro recall", "Macro F1"])
ax.set_ylim(0, comparison[metrics].astype(float).to_numpy().max() * 1.18)
ax.set_title("Test-set performance")
ax.grid(axis="x", visible=False)
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
sns.despine(left=True)
save_fig(fig, "model_comparison.png")
plt.show()

### 4.4 Misclassified test images (best model)

One random misclassified image from each of five different true classes, with true and predicted labels.

In [ ]:
y_pred_best = predictions[BEST]
scores_best = class_scores(models[BEST], X_test)
wrong = np.flatnonzero(y_pred_best != y_test)

rng = np.random.default_rng(SEED)
picks = []
for cls_id in rng.permutation(len(CLASS_NAMES)):
    candidates = wrong[y_test[wrong] == cls_id]
    if len(candidates):
        picks.append(int(rng.choice(candidates)))
    if len(picks) == 5:
        break

fig, axes = plt.subplots(1, len(picks), figsize=(2.7 * len(picks), 3.4), squeeze=False, layout="constrained")
for ax, i in zip(axes[0], picks):
    ax.imshow(load_gray(test_paths[i]), cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"true: {CLASS_NAMES[y_test[i]]}\npred: {CLASS_NAMES[y_pred_best[i]]} "
                 f"({scores_best[i].max():.2f})", fontsize=10)
    ax.set_xlabel(test_paths[i].name, fontsize=7, color=C_MUTED)
    clean_ax(ax)
fig.suptitle(f"Misclassified test images: {BEST} ({len(wrong):,} of {len(y_test):,} test images wrong)")
save_fig(fig, "misclassified_examples.png")
plt.show()

### 4.5 Per-class metrics and most common confusions

The numbers behind the insights below.

In [ ]:
per_class = {}
for name in models:
    p, r, f, s = precision_recall_fscore_support(y_test, predictions[name], labels=range(len(CLASS_NAMES)),
                                                 zero_division=0)
    per_class[name] = pd.DataFrame({"precision": p, "recall": r, "f1": f}, index=CLASS_NAMES)
per_class = pd.concat(per_class, axis=1)
per_class["support"] = np.bincount(y_test, minlength=len(CLASS_NAMES))
per_class.index.name = "class"
per_class.round(3).to_csv(OUTPUT_DIR / "per_class_metrics.csv")
print("Per-class test metrics")
display(per_class.round(3))

cm_best = confusion_matrix(y_test, y_pred_best, labels=range(len(CLASS_NAMES)))
pairs = [(CLASS_NAMES[t], CLASS_NAMES[p], cm_best[t, p], cm_best[t, p] / cm_best[t].sum())
         for t in range(len(CLASS_NAMES)) for p in range(len(CLASS_NAMES)) if t != p]
top_confusions = (pd.DataFrame(pairs, columns=["true", "predicted as", "count", "share of true class"])
                  .sort_values("count", ascending=False).head(5).reset_index(drop=True))
top_confusions.round(3).to_csv(OUTPUT_DIR / "top_confusions.csv", index=False)
print(f"Most common confusions: {BEST}")
display(top_confusions.round(3))

### 4.6 Key insights

> **Placeholder:** these bullets are written from the actual numbers in 4.1–4.5 once the full-dataset run (`SAMPLE_PER_CLASS = None`) has been executed.

## 5. Simple deployment

`predict_emotion` reuses exactly the functions that produced the training features
(`load_gray → preprocess_for_cnn → embed`) and the saved scikit-learn pipeline (scaler + classifier).
Confidence is the predicted-class probability for Logistic Regression. `LinearSVC` has no `predict_proba`, so for
the SVM it is a softmax over the decision scores: a relative confidence, not a calibrated probability.

### 5.1 Save the best model

In [ ]:
MODEL_PATH = MODELS_DIR / "fer_best_model.joblib"
joblib.dump({
    "pipeline": models[BEST],
    "model_name": BEST,
    "class_names": CLASS_NAMES,
    "feature_extractor": "MobileNetV2 (ImageNet), include_top=False, pooling='avg', 224×224×3",
    "preprocessing": {"detector": DETECTOR_NAME, "detect_size": DETECT_SIZE, "detect_margin": DETECT_MARGIN,
                      "crop_pad": CROP_PAD, "cnn_size": CNN_SIZE},
    "test_metrics": results[BEST],
}, MODEL_PATH)

deployed = joblib.load(MODEL_PATH)  # reload from disk so the demo uses the saved artefact
print(f"Saved {deployed['model_name']} → {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1e3:.0f} KB)")

### 5.2 `predict_emotion(image_path) -> (label, confidence)`

In [ ]:
def predict_emotion(image_path, bundle=deployed):
    """Predict the emotion of the face in an image file. Returns (label, confidence)."""
    x, _ = preprocess_for_cnn(load_gray(image_path))
    scores = class_scores(bundle["pipeline"], embed(x[None]))[0]
    k = int(np.argmax(scores))
    return bundle["class_names"][bundle["pipeline"].classes_[k]], float(scores[k])

### 5.3 Demo on unseen images

Uses the images in `samples/` if that folder exists, otherwise one random test image from each of four
different classes (test images were never used for training or model selection).

In [ ]:
SAMPLES_DIR = Path("samples")
sample_files = list_images(SAMPLES_DIR) if SAMPLES_DIR.is_dir() else []
if sample_files:
    demo = [(path, None, None) for path in sample_files[:8]]
    source = "samples/"
else:
    rng = np.random.default_rng(SEED)
    demo = []
    for cls_id in rng.choice(len(CLASS_NAMES), size=4, replace=False):
        i = int(rng.choice(np.flatnonzero(y_test == cls_id)))
        demo.append((test_paths[i], CLASS_NAMES[cls_id], i))
    source = "the test split"

rows = []
fig, axes = plt.subplots(1, len(demo), figsize=(3 * len(demo), 3.8), squeeze=False, layout="constrained")
for ax, (path, true_label, test_idx) in zip(axes[0], demo):
    label, confidence = predict_emotion(path)
    row = {"file": path.name, "predicted": label, "confidence": round(confidence, 3)}
    if true_label is not None:
        row.update({"true": true_label, "matches batch evaluation": label == CLASS_NAMES[y_pred_best[test_idx]]})
    rows.append(row)

    ax.imshow(load_gray(path), cmap="gray", vmin=0, vmax=255)
    title = f"{path.name}\npredicted: {label} ({confidence:.2f})"
    ax.set_title(title + (f"\ntrue: {true_label}" if true_label else ""), fontsize=9)
    clean_ax(ax)
fig.suptitle(f"predict_emotion() with the saved {deployed['model_name']} model, images from {source}")
save_fig(fig, "deployment_demo.png")
plt.show()
display(pd.DataFrame(rows))

---
## Deliverables: where each report item comes from

| Report item | Notebook section | Saved file |
|---|---|---|
| Dataset overview (counts and chart) | 0.1 | `outputs/class_counts.csv`, `outputs/class_distribution.png` |
| Sample faces per class | 0.2 | `outputs/sample_grid.png` |
| Face detection visual check | 1.3 | `outputs/m1_face_detection_grid.png` |
| Detection rate | 1.4 | `outputs/m1_detection_rate.csv` |
| Train–test split verification (shapes, label encoding) | 2.3 | `outputs/label_encoding.csv` |
| Class-weight selection | 3.1 | `outputs/class_weight_selection.csv` |
| Model summaries (preprocessing, hyperparameters, time) | 3.3 | cell output |
| Accuracy, macro-F1, classification reports | 4.1, 4.2 | `outputs/classification_report_*.txt` |
| Confusion matrices (counts and normalised) | 4.1, 4.2 | `outputs/confusion_matrix_*.png` |
| Model comparison | 4.3 | `outputs/model_comparison.csv`, `outputs/model_comparison.png` |
| Misclassified examples | 4.4 | `outputs/misclassified_examples.png` |
| Per-class metrics, top confusions, insights | 4.5, 4.6 | `outputs/per_class_metrics.csv`, `outputs/top_confusions.csv` |
| Deployment demo | 5.3 | `outputs/deployment_demo.png`, `models/fer_best_model.joblib` |